# **Implementation of the RAG pipeline**

This notebook contains the whole implementation of our RAG pipeline, including the loading of the dataset, the retrieving and generator implementation and the evaluating part. It's necessary to run the Data Preperation notebook first, so that the data has the correct format to execude this code.

Tommaso Giorgini

## Imports and Drive mount.

In [ ]:
!pip install -q datasets tqdm requests rank_bm25 \
    "transformers" "accelerate" "bitsandbytes" \
    "huggingface_hub>=0.24.0"
!pip install -q sentence-transformers

from google.colab import drive

from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Any, Optional, Union

import os
import re
import json
import hashlib
import pickle
import subprocess


import numpy as np
import torch
import requests
from tqdm import tqdm


from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

login("") # Secret code for hugging face login

drive.mount("/content/drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.0 MB/s eta 0:00:00
Mounted at /content/drive


## Load Dataset from the Drive (Splits + Docstore)




The dataset is downloaded, extracted and processed in the Data.ipynb notebook.

In [ ]:
BASE = Path("/content/drive/MyDrive/datasets/triviaqa_wiki")
HF_SPLITS = BASE / "hf" / "splits"
HF_DOCS   = BASE / "hf" / "docstore"

ds = load_from_disk(str(HF_SPLITS))           # train/validation/test
docs = load_from_disk(str(HF_DOCS))           # {doc_id, text}


with open(HF_DOCS / "doc_index.json","r") as f:
    DOC_IDX = json.load(f)

def get_doc_text(doc_id: str) -> str:
    i = DOC_IDX.get(doc_id)
    return "" if i is None else docs[int(i)]["text"]

ds

FileNotFoundError: Directory /content/drive/MyDrive/datasets/triviaqa_wiki/hf/splits not found

## RETRIEVER IMPLEMENTATION (SPARSE - BM25)

In [ ]:
# Cache folder for per-document passage lists (so we chunk each doc only once).
CACHE_PASSAGES = Path("/content/passage_cache")
CACHE_PASSAGES.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DENSE_MODEL_NAME   = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=DEVICE)
reranker    = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)


# ======================================================================
# PASSAGE BUILDING

# tokenizer: lowercase + keep alphanumerics/underscore.
# Using a precompiled regex speeds up repeated calls.
_word = re.compile(r"[A-Za-z0-9_]+", flags=re.UNICODE)

def tok(s: str):
    return _word.findall(s.lower())

def norm(s: str):
    return " ".join(_word.findall(s.lower()))

def contains_alias(text: str, aliases):
    T = " " + norm(text) + " "
    for a in (aliases or []):
        A = " " + norm(a) + " "
        if A.strip() and A in T:
            return True
    return False


# Chunking
def chunk_text(text: str, chunk_words=350, overlap=80):
    words = text.split()
    out, i = [], 0
    step = max(1, chunk_words - overlap)  # Positive step ensures overlap.
    while i < len(words):
        out.append(" ".join(words[i:i+chunk_words]))
        i += step
    return out

# Create all passages from the evidence files
def build_passages_for_doc_ids(doc_ids, chunk_words=350, overlap=80):
    out = []
    for doc_id in (doc_ids or []):
        if doc_id not in DOC_IDX:
            continue

        # Per-document cache key (stable and short)
        h = hashlib.md5(doc_id.encode()).hexdigest()
        cf = CACHE_PASSAGES / f"{h}.pkl"

        #if it's already cached
        if cf.exists():
            #Load chunked passages from disk
            passages = pickle.load(open(cf, "rb"))
        else:
            #Read doc text once, chunk it, attach tokenized text
            txt = get_doc_text(doc_id) or ""
            chs = chunk_text(txt, chunk_words, overlap)
            passages = [{
                "pid":   f"{doc_id}:::{j}",
                "doc_id": doc_id,
                "text":  ch,
                "tokens": tok(ch)
            } for j, ch in enumerate(chs)]
            pickle.dump(passages, open(cf, "wb"))  # Save to disk for reuse

        out.extend(passages)
    return out


# To compute Recall@: a passage is labeled true if it contains one of the aliases of the question's answer
def label_passages(passages, answer_aliases):
    return np.array([contains_alias(p["text"], answer_aliases) for p in passages], dtype=bool)



# ======================================================================
# BM25 SPARSE RETRIEVAL

def bm25_search(question, passages, topk=25):

    bm25 = BM25Okapi([p["tokens"] for p in passages])
    # Score against the tokenized question
    scores = bm25.get_scores(tok(question))

    # Take top-k passage indices by descending score
    idx = np.argsort(-scores)[:topk]

    # Return a list of (passage_idx, score) pairs
    return [(int(i), float(scores[i])) for i in idx]



# ======================================================================
# DENSE RETRIEVAL

def dense_search(question, passages, model, topk=30, batch_size=64):

    if len(passages) == 0:
        return []

    texts = [p["text"] for p in passages]

    # Encode passages (batched)
    passage_embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    #Encode question
    query_emb = model.encode(
        question,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Cosine similarity = dot product (since normalized)
    scores = (passage_embs @ query_emb).cpu().numpy()

    idx = np.argsort(-scores)[:topk]
    return [(int(i), float(scores[i])) for i in idx]



# ======================================================================
# RERANKER WITH CROSSENCODER

def rerank_hits_with_crossencoder(
    question,
    passages,
    hits,
    reranker,
    topk=50,
    batch_size=32,
):
    if len(hits) == 0:
        return []

    # Prepare [question, passage] pairs for the cross-encoder
    pair_texts = [
        (question, passages[i]["text"])
        for i, _ in hits
    ]

    # CrossEncoder.predict returns a relevance score per pair
    scores = reranker.predict(pair_texts, batch_size=batch_size)

    # Rebuild (idx, score) and sort
    idxs = [i for i, _ in hits]
    scored = list(zip(idxs, scores))
    scored_sorted = sorted(scored, key=lambda x: x[1], reverse=True)

    return [(int(i), float(s)) for i, s in scored_sorted[:topk]]


# ======================================================================
#Recall at k is the metric for evaluation

def recall_at_k(hits, labels, k=30):
    idxs = [i for i, _ in hits[:k]]
    return float(np.any(labels[idxs])) if len(idxs) > 0 else 0.0



# ======================================================================
# DRIVER

def retrieve_one(
    example,
    topk=30,
    chunk_words=350,
    overlap=70,
    method="bm25",           # "bm25", "dense", or "bm25+rerank"
    dense_model=None,
    reranker=None,
    rerank_candidates=50,    # how many passages to consider before reranking
):

    # Get evidence's files idxs
    doc_ids = example.get("Filenames")
    # Build passages
    passages = build_passages_for_doc_ids(doc_ids, chunk_words, overlap)

    if len(passages) == 0:
        return 0.0, []

    #true if it contain an alias
    labels = label_passages(passages, example.get("Aliases") or [])
    question = example.get("Question")

    #Choose retrieval method
    if method == "bm25":
        hits = bm25_search(question, passages, topk=topk)

    elif method == "dense":
        if dense_model is None:
            raise ValueError("dense_model must be provided for method='dense'")
        hits = dense_search(question, passages, dense_model, topk=topk)

    elif method == "bm25+rerank":
        if reranker is None:
            raise ValueError("reranker must be provided for method='bm25+rerank'")

        #1. BM25 to get a larger set of candidates
        bm25_hits = bm25_search(question, passages, topk=rerank_candidates)

        #2. Rerank with cross-encoder and keep topk
        hits = rerank_hits_with_crossencoder(
            question=question,
            passages=passages,
            hits=bm25_hits,
            reranker=reranker,
            topk=topk,
        )
    else:
        raise ValueError(f"Unknown retrieval method: {method}")

    #Compute Recall@k
    R = recall_at_k(hits, labels, k=topk)

    #Materialize top-k passages
    top_idxs = [i for i, _ in hits[:topk]]
    top_passages = [passages[i] for i in top_idxs]

    return R, top_passages

# ======================================================================
# EVALUATION

def eval_split(
    ds_split,
    topk=30,
    maxN=1000,
    chunk_words=350,
    overlap=80,
    verbose_every=0,
    method="bm25",
    dense_model=None,
    reranker=None,
    rerank_candidates=50,
):
    N = min(maxN, len(ds_split)) if maxN is not None else len(ds_split)
    scores = []

    for i in range(N):
        R, _ = retrieve_one(
            ds_split[i],
            topk=topk,
            chunk_words=chunk_words,
            overlap=overlap,
            method=method,
            dense_model=dense_model,
            reranker=reranker,
            rerank_candidates=rerank_candidates,
        )
        scores.append(R)

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"[{method}] {i+1}/{N}  running Recall@{topk}: {np.mean(scores):.4f}")

    return float(np.mean(scores)), N

TESTING DIFFERENT RETRIVIAL METHODS

In [ ]:
# 1) BM25

dev_ds = ds["validation"]
topk = [1,3,5,7,10,20]
for i in topk:
  print("Testing Recall@",i,": \n")
  bm25_rec, N = eval_split(dev_ds, topk=i, maxN=7900, method="bm25", verbose_every=0)
  print("-BM25 Recall@", i,":", bm25_rec)

# 2) Dense retriever
  dense_rec, N = eval_split(
    dev_ds,
    topk=i,
    maxN=7900,
    method="dense",
    dense_model=dense_model,
    verbose_every=0,
  )
  print("-Dense Recall@",i,":", dense_rec)

# 3) BM25 + reranker
  rerank_rec, N = eval_split(
    dev_ds,
    topk=i,
    maxN=7900,
    method="bm25+rerank",
    reranker=reranker,
    rerank_candidates=30,
    verbose_every=0,
  )
  print("-BM25+Reranker Recall@", i,":", rerank_rec)
  print("\n")

# Generator Implementation

In [ ]:
# ======================================================================
# MODEL SELECTION & LOAD FUNCTION

@dataclass
class GeneratorConfig:
    name: str                     # model name
    hf_id: str                    # HuggingFace model ID
    load_in_4bit: bool = True     # load with 4-bit quantization
    max_new_tokens: int = 64      # maximum number of generated tokens
    temperature: float = 0.0      # 0.0 = deterministic
    top_p: float = 1.0

GENERATOR_MODELS = {
    "phi3": GeneratorConfig(
        name="phi3",
        hf_id="microsoft/Phi-3-mini-4k-instruct",
        load_in_4bit=True,
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
    ),
    "gemma": GeneratorConfig(
        name="gemma",
        hf_id="google/gemma-2b-it",
        load_in_4bit=True,
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
    ),
    "mistral_v03": GeneratorConfig(
        name="mistral_v03",
        hf_id="mistralai/Mistral-7B-Instruct-v0.3",
        load_in_4bit=True,
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
    ),

    "llama_7b": GeneratorConfig(
        name="llama_7b",
        hf_id="meta-llama/Llama-2-7b-chat-hf",
        load_in_4bit=True,
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
    ),

    "qwen2_1_5b": GeneratorConfig(
        name="qwen2_1_5b",
        hf_id="Qwen/Qwen2-1.5B-Instruct",
        load_in_4bit=True,
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
    ),
    "qwen3_8b": GeneratorConfig(
      name="qwen3_8b",
      hf_id="Qwen/Qwen3-8B",
      load_in_4bit=True,
      max_new_tokens=64,
      temperature=0.0,
      top_p=1.0,
),
}


# Global cache (so models are loaded only once)
_LOADED_MODELS: Dict[str, Dict[str, Any]] = {}


def load_generator(model_key: str = "phi3"):

    if model_key not in GENERATOR_MODELS:
        raise ValueError(f"Unknown model '{model_key}'. Available: {list(GENERATOR_MODELS.keys())}")

    # If already loaded then return from cache
    if model_key in _LOADED_MODELS:
        entry = _LOADED_MODELS[model_key]

        return entry["config"], entry["tokenizer"], entry["model"]

    cfg = GENERATOR_MODELS[model_key]

    device_map = "auto"
    torch_dtype = torch.float16

    # 4-bit quantization
    quant_kwargs = {}
    if cfg.load_in_4bit:

        quant_kwargs = {
            "load_in_4bit": True,
            "bnb_4bit_compute_dtype": torch.float16,
            "bnb_4bit_use_double_quant": True,
            "bnb_4bit_quant_type": "nf4",
        }
    else:
        print("[load_generator] Loading model in full precision (no 4-bit quantization).")

    # tokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg.hf_id)

    # Some models require a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg.hf_id,
        device_map=device_map,
        torch_dtype=torch_dtype,
        **quant_kwargs
    )

    # Put model in eval mode (Inference only)
    model.eval()
    print(f"[load_generator] Model '{model_key}' loaded. Device map: {device_map}")

    _LOADED_MODELS[model_key] = {
        "config": cfg,
        "tokenizer": tokenizer,
        "model": model,
    }

    return cfg, tokenizer, model

## Prompt Build

In [ ]:
# ============================================================
# PROMPT VARIANTS FOR TRIVIAQA

def build_prompt_v1(context_block: str, question: str) -> str:
    # Zero shot with strict rules
    return (
        "You answer ONE trivia question at a time.\n"
        "Follow these rules STRICTLY:\n"
        "1. Use ONLY the information in the passages.\n"
        "2. Output ONLY the final answer.\n"
        "3. The answer must be ONE or TWO WORDS.\n"
        "4. Do NOT write full sentences.\n"
        "5. Do NOT repeat or paraphrase the question.\n"
        "6. Do NOT add explanations or extra words.\n"
        "7. Do NOT write any new questions.\n"
        "8. If the answer is not in the passages, output exactly: I don't know\n\n"
        "Passages:\n"
        f"{context_block}\n\n"
        f"Question: {question}\n\n"
        "Final answer (ONE OR TWO WORDS ONLY):"
    )


def build_prompt_v2(context_block: str, question: str) -> str:
    # Few shot
    return (
        "You answer trivia questions based on passages. "
        "Always respond with ONE or TWO WORDS only.\n\n"
        "Examples:\n"
        "Passages:\n"
        "Paris is the capital and most populous city of France.\n"
        "Question: What is the capital of France?\n"
        "Answer: Paris\n\n"
        "Passages:\n"
        "Albert Einstein developed the theory of relativity.\n"
        "Question: Who developed the theory of relativity?\n"
        "Answer: Einstein\n\n"
        "Passages:\n"
        "Ferdinand Magellan led the first expedition to circumnavigate the globe.\n"
        "Question: Who led the first expedition to sail around the world?\n"
        "Answer: Magellan\n\n"
        "Now answer the next question in the SAME FORMAT.\n\n"
        "Passages:\n"
        f"{context_block}\n\n"
        f"Question: {question}\n"
        "Answer (ONE OR TWO WORDS ONLY):"
    )

def build_prompt_v3(context_block: str, question: str) -> str:
    # Span-extraction
    return (
        "You answer trivia questions using the passages below.\n"
        "Your answer must be a short span (ONE or TWO WORDS) copied from the passages.\n"
        "Choose the shortest, most natural alias (e.g., Tolkien, Magellan, New York).\n"
        "Do NOT invent new words.\n"
        "Do NOT explain your answer.\n"
        "If the answer is not present in the passages, output exactly: I don't know\n\n"
        "Passages:\n"
        f"{context_block}\n\n"
        f"Question: {question}\n\n"
        "Final answer (ONE OR TWO WORDS, copied from the passages):"
    )

PROMPT_BUILDERS = {
    "v1": build_prompt_v1,
    "v2": build_prompt_v2,
    "v3": build_prompt_v3,
}


def build_prompt(context_block: str, question: str, variant: str = "v1") -> str:

    if variant not in PROMPT_BUILDERS:
        raise ValueError(f"Unknown prompt variant '{variant}'. Available: {list(PROMPT_BUILDERS.keys())}")
    return PROMPT_BUILDERS[variant](context_block, question)


def build_chat_inputs(
    tokenizer,
    question: str,
    passages: List[str],
    max_passages: int = 5,
    prompt_variant: str = "v1",
):

    selected_passages = passages[:max_passages]
    context_block = "\n\n".join(selected_passages)

    prompt = build_prompt(context_block, question, variant=prompt_variant)

    enc = tokenizer(prompt, return_tensors="pt")
    return enc

## Generate Answer

In [ ]:
def normalize_model_answer(raw: str) -> str:
    line = raw.strip().splitlines()[0]
    line = line.strip(" \"'")
    line = re.sub(r"[.,!?;:]+$", "", line)
    return line


@torch.no_grad()
def generate_answer(
    tokenizer,
    model,
    question: str,
    passages: List[str],
    max_passages: int = 5,
    max_new_tokens: int = 16,
    temperature: float = 0.0,
    top_p: float = 1.0,
    prompt_variant: str = "v1",
) -> str:

    enc = build_chat_inputs(
        tokenizer=tokenizer,
        question=question,
        passages=passages,
        max_passages=max_passages,
        prompt_variant=prompt_variant,
    )

    device = model.device
    enc = {k: v.to(device) for k, v in enc.items()}

    input_ids = enc["input_ids"]
    input_len = input_ids.shape[1]

    gen_ids = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_token_ids = gen_ids[0, input_len:]
    raw_output = tokenizer.decode(new_token_ids, skip_special_tokens=True)

    cleaned = normalize_model_answer(raw_output)
    return cleaned

## End-To-End RAG pipeline

In [ ]:
def rag_answer_one(
    example: Dict[str, Any],
    model_key: str = "phi3",
    retriever_topk: int = 5,
    retriever_type: str = "bm25",   # "bm25", "dense", "bm25_rerank"
    max_passages_for_prompt: int = 25,
    chunk_words: Optional[int] = 350,
    overlap: Optional[int] = 70,
    prompt_variant: str = "v1", # v1, v2 or v3
    dense_model=None,
    reranker=None,
    rerank_candidates=50,
) -> Dict[str, Any]:

    aliases = example.get("Aliases") or []
    if isinstance(aliases, str):
        aliases = [aliases]

    #Load generator
    cfg, tokenizer, model = load_generator(model_key)

    # Chunking parameters
    retrieve_kwargs = {"topk": retriever_topk}
    if chunk_words is not None:
        retrieve_kwargs["chunk_words"] = chunk_words
    if overlap is not None:
        retrieve_kwargs["overlap"] = overlap

    # Retrivial Method chosen
    if retriever_type == "bm25":
        retrieve_kwargs["method"] = "bm25"
    elif retriever_type == "dense":
        retrieve_kwargs["method"] = "dense"
        retrieve_kwargs["dense_model"] = dense_model
    elif retriever_type == "bm25+rerank":
        retrieve_kwargs["method"] = "bm25+rerank"
        retrieve_kwargs["reranker"] = reranker
        retrieve_kwargs["rerank_candidates"] = 50
    else:
        raise ValueError(f"Unknown retriever_type: {retriever_type}")


    # Retrieve Passages
    R, top_passages = retrieve_one(example, **retrieve_kwargs)

    #Compute Recall (0 if alias not present, 1 otherwise)
    print(f"[rag_answer_one] Retrieved {len(top_passages)} passages. Retrieval recall: {R:.3f}")

    passage_texts = [p["text"] for p in top_passages]


    #Generate answer
    answer_text = generate_answer(
        tokenizer=tokenizer,
        model=model,
        question=example.get("Question"),
        passages=passage_texts,
        max_passages=max_passages_for_prompt,
        max_new_tokens=cfg.max_new_tokens,
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        prompt_variant = prompt_variant
    )

    #Build result
    result = {
        "Question": example.get("Question"),
        "Aliases": aliases,
        "generator_model": cfg.hf_id,
        "model_key": cfg.name,
        "retriever_topk": retriever_topk,
        "retrieval_recall": R,
        "retrieved_passages": passage_texts,
        "generated_answer_raw": answer_text,
        "prompt_variant": prompt_variant
    }

    print("Question:", result["Question"])
    print("Aliases:", result["Aliases"])
    print("Generated answer:", result["generated_answer_raw"])
    print("\n")

    return result

# Evaluation

In [ ]:
dev_file = "verified-wikipedia-dev.json"
dev_path = BASE / dev_file
SCORER_FOLDER = BASE / "triviaqa"
SCORER_FILENAME = "triviaqa_evaluation.py"

def _run_triviaqa_scorer(dataset_path, prediction_path) -> str:
    cmd = [
        "python",
        SCORER_FILENAME,
        "--dataset_file", str(dataset_path),
        "--prediction_file", str(prediction_path),
    ]
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True,
            cwd=SCORER_FOLDER
        )
        print("Scorer output:\n", result.stdout)
        return result.stdout
    except subprocess.CalledProcessError as e:
        print(f"Scorer failed with error code {e.returncode}")
        print("Scorer stderr:\n", e.stderr)


def evaluate_triviaqa(
    model_keys: Union[str, List[str]] = "phi3",
    num_examples: int = -1,
    prompt_variants: List[str] = None,
    topk_list: List[int] = None,
    retriever: str = "bm25",
    dense_model=None,
    reranker=None,
    rerank_candidates=50,
    start_index: int = 0,
    end_index: int = None,
):

    if isinstance(model_keys, str):
        model_keys = [model_keys]

    if prompt_variants is None:
        prompt_variants = ["v1", "v2", "v3"]

    if topk_list is None:
        topk_list = [3, 5, 7]

    #Load full dev/train set once
    print(f"Loading ground truth from: {dev_path}")
    with open(dev_path, "r", encoding="utf-8") as f:
        dev_data = json.load(f)

    all_examples = dev_data.get("Data", [])
    total_examples = len(all_examples)
    print(f"Total examples available in file: {total_examples}")

    #select a slice [start_index, end_index) of examples
    if start_index < 0:
        start_index = 0
    if end_index is None or end_index > total_examples:
        end_index = total_examples
    if start_index >= end_index:
        raise ValueError(
            f"Invalid range: start_index ({start_index}) must be < end_index ({end_index})."
        )

    sliced_examples = all_examples[start_index:end_index]

    #Limit the number within that slice
    if num_examples > 0:
        examples_to_evaluate = sliced_examples[:num_examples]
    else:
        examples_to_evaluate = sliced_examples

    print(
        f"Selected original indices: [{start_index}, {start_index + len(examples_to_evaluate)})"
    )
    print(f"Examples selected for evaluation: {len(examples_to_evaluate)}")

    #Nested loops: model x prompt_variant x topk
    for model_key in model_keys:
        for prompt_variant in prompt_variants:
            for topk in topk_list:
                print("\n" + "=" * 80)
                print(f"Model: {model_key} | Retriever: {retriever} | Prompt: {prompt_variant} | topk={topk}")
                print("=" * 80)

                predictions = {}
                processed_examples = []
                num_skipped = 0

                #quick progress print for each example processed
                total_to_process = len(examples_to_evaluate)

                for i, q_info in enumerate(examples_to_evaluate, start=1):
                    print(f"Example {i}/{total_to_process}", end="", flush=True)
                    print()

                    qid = q_info.get("QuestionId", "UNKNOWN_QID")

                    example = {
                        "Question": q_info["Question"],
                        "Filenames": [p["Filename"] for p in q_info.get("EntityPages", [])],
                        "Aliases": q_info["Answer"].get("NormalizedAliases")
                                   or q_info["Answer"].get("Aliases", []),
                    }

                    try:
                        res = rag_answer_one(
                            example,
                            model_key=model_key,
                            retriever_topk=topk,
                            max_passages_for_prompt=topk,
                            prompt_variant=prompt_variant,
                            retriever_type=retriever,
                            dense_model=dense_model,
                            reranker=reranker,
                            rerank_candidates=rerank_candidates,
                        )

                        predicted_answer = (
                            res.get("generated_answer")
                            or res.get("generated_answer_raw", "")
                        )

                        predictions[qid] = predicted_answer
                        processed_examples.append(q_info)

                    except Exception as e:
                        print(
                            f"\nError QID {qid} ({model_key}, {prompt_variant}, "
                            f"topk={topk}, retriever={retriever}): {e}"
                        )
                        num_skipped += 1
                        continue

                num_eval = len(processed_examples)
                print(f"Processed {num_eval} examples (skipped {num_skipped}).")

                if num_eval == 0:
                    print("No successful predictions, skipping scoring.")
                    continue

                # Build subset dataset
                subset_data = dict(dev_data)
                subset_data["Data"] = processed_examples

                ts = datetime.now().strftime("%Y%m%d-%H%M%S")
                meta = subset_data.get("Meta", {})
                meta.update({
                    "model_key": model_key,
                    "retriever": retriever,
                    "prompt_variant": prompt_variant,
                    "topk": topk,
                    "num_examples": num_eval,
                    "timestamp": ts,
                    "start_index": start_index,
                    "end_index": start_index + len(examples_to_evaluate),
                })
                subset_data["Meta"] = meta

                subset_dir = BASE / "eval_subsets"
                subset_dir.mkdir(parents=True, exist_ok=True)
                subset_path = subset_dir / (
                    f"triviaqa_dev_subset_{model_key}_{retriever}_{prompt_variant}_topk{topk}_"
                    f"{num_eval}ex_{ts}.json"
                )

                with open(subset_path, "w", encoding="utf-8") as f:
                    json.dump(subset_data, f, ensure_ascii=False, indent=2)

                #Save predictions
                pred_dir = BASE / "predictions"
                pred_dir.mkdir(parents=True, exist_ok=True)
                pred_path = pred_dir / (
                    f"triviaqa_dev_pred_{model_key}_{retriever}_{prompt_variant}_topk{topk}_"
                    f"{num_eval}ex_{ts}.json"
                )

                with open(pred_path, "w", encoding="utf-8") as f:
                    json.dump(predictions, f, ensure_ascii=False, indent=2)

                print(f"Subset saved to:      {subset_path}")
                print(f"Predictions saved to: {pred_path}")

                #Run scorer and SAVE its output
                scorer_output = _run_triviaqa_scorer(subset_path, pred_path)

                scores_dir = BASE / "newscores"
                scores_dir.mkdir(parents=True, exist_ok=True)
                scores_path = scores_dir / (
                    f"triviaqa_scores_{model_key}_{retriever}_{prompt_variant}_topk{topk}_"
                    f"{num_eval}ex_{ts}.txt"
                )

                with open(scores_path, "w", encoding="utf-8") as f:
                    f.write(scorer_output)

                print(f"Scores saved to:      {scores_path}")



In [ ]:
# For testing

evaluate_triviaqa(
    model_keys=["mistral_v03"],
    num_examples=-1,
    prompt_variants=["v1"],
    topk_list=[10],
    retriever="bm25+rerank",
    dense_model=dense_model,
    reranker=reranker,
    rerank_candidates=30,
    start_index=0,
    end_index=7900,
)

